<a href="https://colab.research.google.com/github/mdrs-thiago/mdrs-thiago.github.io/blob/main/topicos_a/2027/notebooks/aula01_demo_xor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 01 · Demo da quest: quem resolve o XOR?
**Tópicos Especiais A · Deep Learning · UERJ 2027.1** · Bloco 1 — Fundamentos

Notebook para rodar **ao vivo** durante a quest *Place your Bets*. Para cada uma das 4 configurações do slide, treinamos com 20 seeds diferentes e contamos quantas chegam a 100% de acurácia no XOR.

Roda em CPU em menos de 1 minuto. Não é preciso GPU.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])
print(X.shape, y.shape)

## As 4 configurações

Todas usam a mesma saída (1 neurônio, com `BCEWithLogitsLoss`) e o otimizador Adam. Só muda o que está entre a entrada e a saída.

In [ ]:
def make_model(config):
    if config == 1:   # perceptron: só uma camada linear
        return nn.Linear(2, 1)
    if config == 2:   # 2 ocultos, sem ativação
        return nn.Sequential(nn.Linear(2, 2), nn.Linear(2, 1))
    if config == 3:   # 2 ocultos + ReLU
        return nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1))
    if config == 4:   # 8 ocultos + ReLU
        return nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1))

CONFIGS = {
    1: dict(nome='Perceptron (sem camada oculta)', lr=0.01, passos=5000),
    2: dict(nome='2 ocultos, sem ativação',         lr=0.01, passos=5000),
    3: dict(nome='2 ocultos + ReLU',                lr=0.01, passos=5000),
    4: dict(nome='8 ocultos + ReLU',                lr=0.01, passos=5000),
}

def treinar(config, seed):
    torch.manual_seed(seed)
    model = make_model(config)
    c = CONFIGS[config]
    opt = torch.optim.Adam(model.parameters(), lr=c['lr'])
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(c['passos']):
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
    with torch.no_grad():
        acc = ((model(X) > 0).float() == y).float().mean().item()
    return model, acc

## Rodando as apostas

Rode uma configuração por vez, **depois** que os grupos apostarem.

In [ ]:
SEEDS = range(20)

def rodar(config):
    accs = [treinar(config, s)[1] for s in SEEDS]
    resolveu = sum(a == 1.0 for a in accs)
    print(f"Config {config} · {CONFIGS[config]['nome']}: resolveu em {resolveu}/{len(accs)} seeds")
    return accs

resultados = {}
resultados[1] = rodar(1)

In [ ]:
resultados[2] = rodar(2)

In [ ]:
resultados[3] = rodar(3)

In [ ]:
resultados[4] = rodar(4)

## Olhando a fronteira de decisão

A configuração 3 é a mais interessante: algumas seeds resolvem e outras não. Vamos comparar uma de cada.

In [ ]:
def plot_fronteira(model, ax, titulo):
    xx, yy = torch.meshgrid(torch.linspace(-0.5, 1.5, 200), torch.linspace(-0.5, 1.5, 200), indexing='xy')
    grid = torch.stack([xx.ravel(), yy.ravel()], dim=1)
    with torch.no_grad():
        zz = (model(grid) > 0).float().reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=['#fde2e2', '#dbeafe'])
    for (a, b), t in zip(X.tolist(), y.ravel().tolist()):
        ax.scatter(a, b, s=200, c='#004b87' if t else '#e53e3e', marker='o' if t else 's', edgecolors='k')
    ax.set_title(titulo); ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

ok   = next(s for s, a in enumerate(resultados[3]) if a == 1.0)
fail = next(s for s, a in enumerate(resultados[3]) if a < 1.0)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
plot_fronteira(treinar(3, ok)[0],   axes[0], f'Config 3, seed {ok}: resolveu')
plot_fronteira(treinar(3, fail)[0], axes[1], f'Config 3, seed {fail}: não resolveu')
plot_fronteira(treinar(4, 0)[0],    axes[2], 'Config 4, seed 0')
plt.tight_layout(); plt.show()

### Para discutir

- Na seed que falhou, olhe a saída da camada oculta: algum neurônio ReLU ficou com saída 0 para os 4 pontos? (um neurônio "morto")
- Por que 8 neurônios quase nunca falham? Gancho para a **Aula 03** (inicialização e learning rate).

In [ ]:
model_fail = treinar(3, fail)[0]
with torch.no_grad():
    h = model_fail[1](model_fail[0](X))   # saída das ReLUs ocultas
print(h)